## Imports

In [1]:
import warnings
warnings.filterwarnings("ignore", category=RuntimeWarning, module="sklearn")
warnings.filterwarnings("ignore", category=RuntimeWarning, module="pydeseq2")

from pathlib import Path

import pandas as pd
import numpy as np
import plotly.express as px

from pydeseq2.dds import DeseqDataSet
from pydeseq2.ds import DeseqStats

## Configurações

In [2]:
PROCESSED_DIR = Path("../../data/interim")
RESULTS_DIR = Path("../../data/interim/deseq2")

FILTERED_EXPRESSION_PATH = PROCESSED_DIR / "microplastic_expression_filtered.csv"
METADATA_PATH = PROCESSED_DIR / "microplastic_metadata.csv"

# Arquivo de entrada para o WGCNA: log2(normed_counts + 1) — não é VST paramétrico
WGCNA_INPUT_PATH = PROCESSED_DIR / "microplastic_log2norm_wgcna_input.csv"

RESULTS_DIR.mkdir(parents=True, exist_ok=True)

ALPHA = 0.05  # Alinhado ao artigo base
LFC_THRESHOLD = 1.0  # usado apenas para coloração do volcano plot — NÃO filtra genes significativos

## Carregamento dos Dados Filtrados

In [3]:
expression_df = pd.read_csv(FILTERED_EXPRESSION_PATH)
metadata_df = pd.read_csv(METADATA_PATH)

print("Matriz filtrada:", expression_df.shape)
print("Metadados:", metadata_df.shape)

display(expression_df.head())
display(metadata_df.head())

Matriz filtrada: (12176, 25)
Metadados: (24, 9)


,gene_id,CTR_1,CTR_2,CTR_3,MA1_1,MA1_2,MA1_3,MB1_1,MB1_2,MB1_3,...,MD1_3,MA100_1,MA100_2,MA100_3,MB100_1,MB100_2,MB100_3,MC100_1,MC100_2,MC100_3
0,ENSG00000000003,357,327,329,276,226,355,258,275,464,...,336,336,285,323,333,233,334,310,358,398
1,ENSG00000000419,770,453,733,605,629,716,448,748,863,...,969,721,700,733,710,703,633,727,974,712
2,ENSG00000000457,27,115,31,26,33,35,23,32,37,...,47,30,31,7,22,40,44,41,68,67
3,ENSG00000000460,37,27,7,4,5,36,14,45,52,...,0,29,3,0,17,9,19,12,21,33
4,ENSG00000001036,980,943,970,807,886,1855,772,1061,1049,...,1156,1094,780,1174,996,903,1046,1286,1353,1381


,particle_type,particle_size_um,particle_size_nm,concentration_gL,is_control,treatment_status,sample_id,group,replicate
0,control,NaN,NaN,0.0,True,control,CTR_1,CTR,1
1,control,NaN,NaN,0.0,True,control,CTR_2,CTR,2
2,control,NaN,NaN,0.0,True,control,CTR_3,CTR,3
3,polystyrene,1.0,1000.0,0.1,False,treated,MA1_1,MA1,1
4,polystyrene,1.0,1000.0,0.1,False,treated,MA1_2,MA1,2


## Preparação da Matriz para DEG (Differentially Expressed Genes)

In [4]:
# A matriz carregada tem genes como linhas e amostras como colunas, mas o DESeq2 espera o contrário.
# Transpondo a matriz para ter amostras como linhas e genes como colunas.

sample_ids = metadata_df["sample_id"].tolist()
expr_sample_cols = [c for c in expression_df.columns if c != "gene_id"]

missing_in_expr = sorted(set(sample_ids) - set(expr_sample_cols))
missing_in_meta = sorted(set(expr_sample_cols) - set(sample_ids))

if missing_in_expr or missing_in_meta:
    raise ValueError(
        f"Inconsistência entre matriz e metadata.\n"
        f"Ausentes na expressão: {missing_in_expr}\n"
        f"Ausentes no metadata: {missing_in_meta}"
    )

# Reordena colunas pela ordem do metadata
expression_df = expression_df[["gene_id"] + sample_ids]

# counts: samples x genes
counts = expression_df.set_index("gene_id").T
counts.index.name = "sample_id"

# Garante inteiros
counts = counts.astype(int)

# Metadata indexado por sample_id
metadata = metadata_df.set_index("sample_id").loc[counts.index].copy()

# Fator de condição principal para os contrastes
metadata["condition"] = metadata["group"].astype(str)

print("Counts:", counts.shape)
print("Metadata:", metadata.shape)

display(counts.iloc[:5, :5])
display(metadata.head())

Counts: (24, 12176)
Metadata: (24, 9)


gene_id,ENSG00000000003,ENSG00000000419,ENSG00000000457,ENSG00000000460,ENSG00000001036
sample_id,,,,,
CTR_1,357,770,27,37,980
CTR_2,327,453,115,27,943
CTR_3,329,733,31,7,970
MA1_1,276,605,26,4,807
MA1_2,226,629,33,5,886


,particle_type,particle_size_um,particle_size_nm,concentration_gL,is_control,treatment_status,group,replicate,condition
sample_id,,,,,,,,,
CTR_1,control,NaN,NaN,0.0,True,control,CTR,1,CTR
CTR_2,control,NaN,NaN,0.0,True,control,CTR,2,CTR
CTR_3,control,NaN,NaN,0.0,True,control,CTR,3,CTR
MA1_1,polystyrene,1.0,1000.0,0.1,False,treated,MA1,1,MA1
MA1_2,polystyrene,1.0,1000.0,0.1,False,treated,MA1,2,MA1


In [5]:
# Validações finais antes do DESeq2

print("Condições disponíveis:")
print(sorted(metadata["condition"].unique()))

print("\nNúmero de amostras por condição:")
display(metadata["condition"].value_counts().sort_index())

print("\nAlgum valor ausente em counts?", counts.isna().sum().sum() > 0)
print("Algum valor negativo em counts?", (counts < 0).any().any())
print("Todos os counts são inteiros?", np.all(np.equal(np.mod(counts.to_numpy(), 1), 0)))

Condições disponíveis:
['CTR', 'MA1', 'MA100', 'MB1', 'MB100', 'MC1', 'MC100', 'MD1']

Número de amostras por condição:


condition
CTR      3
MA1      3
MA100    3
MB1      3
MB100    3
MC1      3
MC100    3
MD1      3
Name: count, dtype: int64


Algum valor ausente em counts? False
Algum valor negativo em counts? False
Todos os counts são inteiros? True


## Ajuste do Modelo DESeq2

In [6]:
dds = DeseqDataSet(
    counts=counts,
    metadata=metadata,
    design_factors="condition",
    quiet=False,
)

dds.deseq2()

print("Modelo DESeq2 ajustado com sucesso.")

Using None as control genes, passed at DeseqDataSet initialization


/tmp/ipykernel_96175/2196543317.py:1: DeprecationWarning: design_factors is deprecated and will soon be removed.Please consider providing a formulaic formula using the design argumentinstead.
  dds = DeseqDataSet(
Fitting size factors...
... done in 0.01 seconds.

Fitting dispersions...
... done in 0.61 seconds.

Fitting dispersion trend curve...
... done in 0.12 seconds.

Fitting MAP dispersions...
... done in 0.73 seconds.

Fitting LFCs...


Modelo DESeq2 ajustado com sucesso.


... done in 0.58 seconds.

Calculating cook's distance...
... done in 0.01 seconds.

Replacing 0 outlier genes.



In [7]:
# NOTA: PyDESeq2 não implementa VST completo. Usamos log2(normed_counts + 1) como aproximação (pseudocontagem).
# O resultado é adequado para o WGCNA, mas pode manter variância maior em genes de alta expressão em comparação
# com um VST baseado no modelo de dispersão.

contagens_normalizadas = dds.layers['normed_counts']
matriz_log2norm = np.log2(contagens_normalizadas + 1)

df_wgcna_input = pd.DataFrame(
    matriz_log2norm,
    index=dds.obs_names,
    columns=dds.var_names
)
df_wgcna_input = df_wgcna_input.T
df_wgcna_input.index.name = "gene_id"
df_wgcna_input = df_wgcna_input.reset_index()

# Remove a coluna 'sample_id' se existir
if 'sample_id' in df_wgcna_input.columns:
    df_wgcna_input = df_wgcna_input.drop(columns=['sample_id'])

print("Matriz log2norm pronta para o WGCNA!")
print("Dimensões:", df_wgcna_input.shape)

df_wgcna_input.to_csv(WGCNA_INPUT_PATH, index=False)

Matriz log2norm pronta para o WGCNA!
Dimensões: (12176, 25)


## Definição dos Contrastes

In [8]:
# Queremos contrastar as seguintes comparações, onde o primeiro elemento é o grupo de teste
# e o segundo é o grupo de referência (controle). O DESeq2 irá comparar o primeiro grupo contra
# o segundo, então a interpretação dos resultados será "diferença do grupo de teste em relação ao
# grupo de referência".

contrasts = [
    # Tratamento vs controle
    ("MA100", "CTR"),
    ("MB100", "CTR"),
    ("MC100", "CTR"),
    ("MA1", "CTR"),
    ("MB1", "CTR"),
    ("MC1", "CTR"),
    ("MD1", "CTR"),

    # Comparações pareadas por concentração entre 0.1 µm e 1 µm
    ("MA100", "MA1"),
    ("MB100", "MB1"),
    ("MC100", "MC1"),
]

print("Contrastes definidos:")
for tested, ref in contrasts:
    print(f"- {tested} vs {ref}")

Contrastes definidos:
- MA100 vs CTR
- MB100 vs CTR
- MC100 vs CTR
- MA1 vs CTR
- MB1 vs CTR
- MC1 vs CTR
- MD1 vs CTR
- MA100 vs MA1
- MB100 vs MB1
- MC100 vs MC1


## Execução dos Contrastes

In [9]:
## Função auxiliar para rodar contraste

def run_contrast(dds, tested_level: str, ref_level: str, alpha: float = 0.05) -> pd.DataFrame:
    """
    Executa um contraste DESeq2 e retorna um DataFrame com os resultados.
    Genes são marcados como `significant` se `padj < alpha` (sem corte de LFC).
    """
    stat_res = DeseqStats(
        dds,
        contrast=["condition", tested_level, ref_level],
        alpha=alpha,
        quiet=True,
    )

    stat_res.summary()

    res = stat_res.results_df.copy().reset_index()
    res = res.rename(columns={"index": "gene_id"})
    res["contrast"] = f"{tested_level}_vs_{ref_level}"

    # Flags úteis
    res["significant"] = (~res["padj"].isna()) & (res["padj"] < alpha)
    res["direction"] = np.where(
        res["log2FoldChange"] > 0,
        f"up_in_{tested_level}",
        f"up_in_{ref_level}"
    )

    return res.sort_values(["padj", "pvalue"], na_position="last")

In [10]:
## Execução dos contrastes

all_results = {}
summary_rows = []

for tested, ref in contrasts:
    contrast_name = f"{tested}_vs_{ref}"
    print(f"\n=== Rodando contraste: {contrast_name} ===")

    res = run_contrast(dds, tested_level=tested, ref_level=ref, alpha=ALPHA)
    all_results[contrast_name] = res

    # Salva tabela individual
    output_file = RESULTS_DIR / f"deg_{contrast_name}.csv"
    res.to_csv(output_file, index=False)

    sig = res["significant"].fillna(False)
    up = sig & (res["log2FoldChange"] > 0)
    down = sig & (res["log2FoldChange"] < 0)

    summary_rows.append({
        "contrast": contrast_name,
        "n_genes_tested": res.shape[0],
        "n_significant": int(sig.sum()),
        "n_up_in_tested": int(up.sum()),
        "n_down_in_tested": int(down.sum()),
        "output_file": str(output_file),
    })

summary_df = pd.DataFrame(summary_rows)
display(summary_df)


=== Rodando contraste: MA100_vs_CTR ===

=== Rodando contraste: MB100_vs_CTR ===

=== Rodando contraste: MC100_vs_CTR ===

=== Rodando contraste: MA1_vs_CTR ===

=== Rodando contraste: MB1_vs_CTR ===

=== Rodando contraste: MC1_vs_CTR ===

=== Rodando contraste: MD1_vs_CTR ===

=== Rodando contraste: MA100_vs_MA1 ===

=== Rodando contraste: MB100_vs_MB1 ===

=== Rodando contraste: MC100_vs_MC1 ===


,contrast,n_genes_tested,n_significant,n_up_in_tested,n_down_in_tested,output_file
0,MA100_vs_CTR,12176,1361,658,703,../../data/interim/deseq2/deg_MA100_vs_CTR.csv
1,MB100_vs_CTR,12176,3,1,2,../../data/interim/deseq2/deg_MB100_vs_CTR.csv
2,MC100_vs_CTR,12176,0,0,0,../../data/interim/deseq2/deg_MC100_vs_CTR.csv
3,MA1_vs_CTR,12176,238,107,131,../../data/interim/deseq2/deg_MA1_vs_CTR.csv
4,MB1_vs_CTR,12176,689,356,333,../../data/interim/deseq2/deg_MB1_vs_CTR.csv
5,MC1_vs_CTR,12176,1481,728,753,../../data/interim/deseq2/deg_MC1_vs_CTR.csv
6,MD1_vs_CTR,12176,1227,575,652,../../data/interim/deseq2/deg_MD1_vs_CTR.csv
7,MA100_vs_MA1,12176,147,92,55,../../data/interim/deseq2/deg_MA100_vs_MA1.csv
8,MB100_vs_MB1,12176,17,8,9,../../data/interim/deseq2/deg_MB100_vs_MB1.csv
9,MC100_vs_MC1,12176,1914,857,1057,../../data/interim/deseq2/deg_MC100_vs_MC1.csv


## Visualizações

In [11]:
def plot_volcano(
    results_df,
    title: str,
    alpha: float = 0.05,
    lfc_threshold: float = 1.0,
    max_points: int = 100000,
):
    df = results_df.copy()

    df = df.dropna(subset=["log2FoldChange", "pvalue"])

    if max_points is not None and df.shape[0] > max_points:
        df = df.sample(max_points, random_state=42)

    df["neglog10_padj"] = -np.log10(df["padj"].fillna(1.0).clip(lower=1e-300))

    df["Status"] = "não significativo"
    df.loc[(df["padj"] < alpha) & (df["log2FoldChange"] >= lfc_threshold), "Status"] = "up"
    df.loc[(df["padj"] < alpha) & (df["log2FoldChange"] <= -lfc_threshold), "Status"] = "down"

    fig = px.scatter(
        df,
        x="log2FoldChange",
        y="neglog10_padj",
        color="Status",
        color_discrete_map={
            "não significativo": "lightgray",
            "up": "red",
            "down": "blue"
        },
        hover_name="gene_id",
        title=title,
        labels={
            "log2FoldChange": "log2 Fold Change",
            "neglog10_padj": "-log10(padj)"
        }
    )

    fig.add_vline(x=lfc_threshold, line_dash="dash", line_color="black")
    fig.add_vline(x=-lfc_threshold, line_dash="dash", line_color="black")
    fig.add_hline(y=-np.log10(alpha), line_dash="dash", line_color="black")

    fig.update_traces(marker=dict(size=6, opacity=0.7))
    fig.update_layout(height=600, width=800)

    fig.show()

In [12]:
plot_volcano(
    all_results["MA100_vs_CTR"],
    title="Volcano Plot — MA100 vs CTR",
    alpha=ALPHA,
    lfc_threshold=LFC_THRESHOLD,
)

In [13]:
# Volcano plots para os 3 contrastes pareados por tamanho de partícula.
# Esses contrastes respondem diretamente à pergunta de pesquisa:
# 0.1 µm (100nm) vs 1 µm na mesma concentração (alta, média, baixa).

for contrast_name in ["MA100_vs_MA1", "MB100_vs_MB1", "MC100_vs_MC1"]:
    plot_volcano(
        all_results[contrast_name],
        title=f"Volcano Plot — {contrast_name} (efeito do tamanho de partícula)",
        alpha=ALPHA,
        lfc_threshold=LFC_THRESHOLD,
    )

## Salvamento dos Resultados

In [14]:
# Consolida todos os resultados em uma única tabela para análises posteriores
# Os resultados reproduzem bem o artigo base, com um número de DEGs na mesma ordem de magnitude, porém
# a quantidade exata de DEGs difere pois filtramos apenas genes codificadores de proteínas antes do DESeq2.

all_results_df = pd.concat(all_results.values(), axis=0, ignore_index=True)

all_results_path = RESULTS_DIR / "deg_all_contrasts.csv"
all_results_df.to_csv(all_results_path, index=False)

print(f"Tabela consolidada salva em: {all_results_path}")
print("Dimensões:", all_results_df.shape)

Tabela consolidada salva em: ../../data/interim/deseq2/deg_all_contrasts.csv
Dimensões: (121760, 10)
